In [1]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local
from kfp.dsl import Input, Output, Dataset, Model, Artifact

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [2]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


In [3]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
)
def finetune_model(base_model: str, dataset_path: str, epochs:int, mlflow_url: str, mlflow_s3_url: str, mlflow_experiment: str,finetuned_model: Output[Model]):
    import subprocess
    subprocess.run(['nvidia-smi'])
    subprocess.run(['pwd'])
    
    from ultralytics import YOLO
    import os
    import time
    
    def update_token(trainer):
        with open('/etc/secrets/ezua/.auth_token','r') as file:
            AUTH_TOKEN = file.read()
            os.environ['MLFLOW_TRACKING_TOKEN']=AUTH_TOKEN
            os.environ["AWS_ACCESS_KEY_ID"] = AUTH_TOKEN
            os.environ["AWS_SECRET_ACCESS_KEY"] = "s3"
            print("Token successfully synced!!")
        
    model = YOLO(base_model)
    model.add_callback("on_model_save",update_token)
    model.callbacks["on_model_save"]

    # MLFLOW configurations
    run_name = model.model_name.name.replace('.pt','-') +time.strftime("%Y%m%d-%H%M%S", time.localtime())
    os.environ['MLFLOW_RUN'] = run_name
    os.environ['MLFLOW_TRACKING_URI'] = mlflow_url
    os.environ['MLFLOW_S3_ENDPOINT_URL'] = mlflow_s3_url
    os.environ['MLFLOW_EXPERIMENT_NAME'] = mlflow_experiment
    os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
    os.environ['MLFLOW_S3_IGNORE_TLS'] = 'true'

    # Set the Token
    update_token('dummy')

    results = model.train(data=dataset_path, epochs=epochs)  # train the model
    model_path = results.save_dir.joinpath("weights/best.pt")

    subprocess.run(['cp',model_path,finetuned_model.path])
    print('Information about the artifact')
    print('Name:', finetuned_model.name)
    print('URI:', finetuned_model.uri)
    print('Path:', finetuned_model.path)
    print('Metadata:', finetuned_model.metadata)

In [4]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
)
def benchmark_model(finetuned_model: Input[Model], dataset_path: str, benchmark_result: Output[Artifact]):
    import subprocess
    subprocess.run(['nvidia-smi'])
    subprocess.run(['pwd'])
    
    from ultralytics import YOLO
    from ultralytics.utils.benchmarks import benchmark
    import pandas as pd
    import os
    import time
    import torch
    import pandas
    
    test_formats = ['-','torchscript','onnx','engine']

    # Copy model file into YOLO compatible name. --.pt
    subprocess.run(['cp',finetuned_model.path,finetuned_model.path + '.pt'])

    model_path = finetuned_model.path + '.pt'
    all_result = pd.DataFrame(columns=['Format','Status❔', 'Size (MB)', 'metrics/mAP50-95(B)', 'Inference time (ms/im)', 'FPS'])

    for format in test_formats:
        result = benchmark(model=model_path, data=dataset_path, device=0,format=format)
        half_result = benchmark(model=model_path, data=dataset_path, device=0,format=format,half=True)
        half_result['Format'] = half_result['Format'] + '_half'
        
        all_result = pd.concat([all_result,result],ignore_index=True)
        all_result = pd.concat([all_result,half_result],ignore_index=True)

    print(all_result)
    all_result.to_csv(benchmark_result.path)

In [5]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
    packages_to_install=[]
)
def export_license_detect_model(finetuned_model: Input[Model], engine: Output[Model], license_config: Output[Artifact]):
    import subprocess
    # Copy model file into YOLO compatible name. --.pt
    subprocess.run(['cp',finetuned_model.path,finetuned_model.path + '.pt'])

    from ultralytics import YOLO
    import torch
    # Retrieve metadata during export. Metadata needs to be added to config.pbtxt. See next section.
    metadata_license = []
    def export_cb_for_license(exporter):
        metadata_license.append(exporter.metadata)

    license_detector = YOLO(finetuned_model.path + '.pt')
    license_detector.add_callback("on_export_end", export_cb_for_license)
    
    # Export the model
    license_detector_engine = license_detector.export(format="engine", dynamic=True,half=True,device=0,nms=True)
    subprocess.run(['cp',license_detector_engine,engine.path])
    # change Dimensions properly
    data = """
# Add metadata
parameters {
  key: "metadata"
  value {
    string_value: "%s"
  }
}

name: "license_detector"
platform: "tensorrt_plan"
max_batch_size : 0
input [
  {
    name: "images"
    dims: [ -1, 3, -1, -1 ]
  }
]
output [
  {
    name: "output0"
    dims: [ -1, 300, 6 ]
  }
]
""" % metadata_license[0]
    
    with open(license_config.path, "w") as f:
        f.write(data)

In [6]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
    packages_to_install=[]
)
def export_vehicle_detect_model(base_model: str, engine: Output[Model], vehicle_config: Output[Artifact]):
    from ultralytics import YOLO
    import torch
    import subprocess
    
    # Retrieve metadata during export. Metadata needs to be added to config.pbtxt. See next section.
    metadata_vehicle = []
    def export_cb_for_vehicle(exporter):
        metadata_vehicle.append(exporter.metadata)

    vehicle_detector = YOLO(base_model)
    vehicle_detector.add_callback("on_export_end", export_cb_for_vehicle)
    
    # Export the model
    vehicle_detector_engine = vehicle_detector.export(format="engine", dynamic=True,half=True,device=0,nms=True)
    subprocess.run(['cp',vehicle_detector_engine,engine.path])
    # change Dimensions properly
    data = """
# Add metadata
parameters {
  key: "metadata"
  value {
    string_value: "%s"
  }
}

name: "vehicle_detector"
platform: "tensorrt_plan"
max_batch_size : 0
input [
  {
    name: "images"
    dims: [ -1, 3, -1, -1 ]
  }
]
output [
  {
    name: "output0"
    dims: [ -1, 300, 6 ]
  }
]
""" % metadata_vehicle[0]

    with open(vehicle_config.path, "w") as f:
        f.write(data)

In [7]:
@dsl.component(
    base_image='geuntakroh/kfp-test:v0.9',
)
def upload_engines(
    mlflow_url: str,
    mlflow_s3_url: str,
    license_engine: Input[Model],
    license_config: Input[Artifact],
    vehicle_engine: Input[Model],
    vehicle_config: Input[Artifact]
) -> str:
    from pathlib import Path
    import shutil
    import os
    import time
    import subprocess

    base = Path("triton_engines")
        
    # Model configurations
    models = [
        ("license_detector", license_config.path, license_engine.path, 1),
        ("vehicle_detector", vehicle_config.path, vehicle_engine.path, 1)
    ]
    
    for model_name, config_file, engine_file,version in models:
        # Create directory structure
        model_dir = base / model_name
        version_dir = model_dir / str(version)
        version_dir.mkdir(parents=True, exist_ok=True)
        
        # Move files
        shutil.copy(config_file, model_dir / "config.pbtxt")
        shutil.copy(engine_file, version_dir / "model.plan")
    
    def display_tree(directory, prefix="", is_last=True):
        """Display directory tree using pathlib"""
        directory = Path(directory)
        print(prefix + ("└── " if is_last else "├── ") + directory.name)
        
        if directory.is_dir():
            children = sorted(directory.iterdir())
            for i, child in enumerate(children):
                is_last_child = i == len(children) - 1
                new_prefix = prefix + ("    " if is_last else "│   ")
                display_tree(child, new_prefix, is_last_child)
    
    path = Path("triton_engines")
    display_tree(path)

    # Logging Engine Artifacts at MLFlow
    os.environ['MLFLOW_TRACKING_URI'] = mlflow_url
    os.environ['MLFLOW_S3_ENDPOINT_URL'] = mlflow_s3_url
    with open('/etc/secrets/ezua/.auth_token','r') as file:
        AUTH_TOKEN = file.read()
        os.environ['MLFLOW_TRACKING_TOKEN']=AUTH_TOKEN
        os.environ["AWS_ACCESS_KEY_ID"] = AUTH_TOKEN
        os.environ["AWS_SECRET_ACCESS_KEY"] = "s3"
        
    result = subprocess.run(
        ['python',
         'mlflow_scripts/publish_model_to_mlflow.py',
         '--model_name',
         'license-detectors',
         '--model_directory',
         './triton_engines',
         '--flavor',
         'triton'],capture_output=True,text=True)
    s3_uri = "s3://" + result.stdout.split("s3://")[1].strip() + '/triton/triton_engines'
    print(s3_uri)
    return s3_uri

In [11]:
@dsl.pipeline(
    name="prepare_model_pipe"
)
def prepare_model_pipe(
    base_model: str,
    # datasets: str,
    datasets_pvc_name: str,
    mlflow_url: str,
    mlflow_s3_url: str,
    mlflow_experiment: str,
    epochs: int,
) -> str:
    target_path = '/data'
    dataset_path = target_path + '/data.yaml' # /data/data.yaml
    
    # Fine-tuning the Model to detect license plate
    task1 = finetune_model(
        base_model=base_model,
        dataset_path=dataset_path,
        epochs=epochs,
        mlflow_url=mlflow_url,
        mlflow_s3_url=mlflow_s3_url,
        mlflow_experiment=mlflow_experiment
    )
    task1.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit("1")
    task1.set_cpu_limit("4")
    task1.set_memory_limit("16Gi")
    kubernetes.mount_pvc(
        task1,
        pvc_name=datasets_pvc_name,
        mount_path=target_path,
    )
    # Increase /dev/shm. Use emptydir beyond the KFP backend 2.3
    dev_shm = kubernetes.CreatePVC(
        pvc_name='mimic-dev-shm',
        access_modes=['ReadWriteMany'],
        size='5Gi',
        storage_class_name=current_sc, # gl4fs-system for PCAI
    )
    kubernetes.mount_pvc(
        task1,
        pvc_name='mimic-dev-shm',
        mount_path='/dev/shm',
    )
    kfp.kubernetes.add_pod_annotation(
        task=task1,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    
    # Benchmark Each model format
    task2 = benchmark_model(finetuned_model=task1.outputs['finetuned_model'],dataset_path=dataset_path)
    task2.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit("1")
    task2.set_cpu_limit("4")
    task2.set_memory_limit("16Gi")
    kfp.kubernetes.add_pod_annotation(
        task=task2,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    kubernetes.mount_pvc(
        task2,
        pvc_name=datasets_pvc_name,
        mount_path=target_path,
    )

    # Export TensorRT engine from fine-tuned model
    task3 = export_license_detect_model(finetuned_model=task1.outputs['finetuned_model'])
    task3.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit("1")
    task3.set_cpu_limit("4")
    task3.set_memory_limit("16Gi")
    kfp.kubernetes.add_pod_annotation(
        task=task3,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    # Export TensorRT Engine from pre-trained model
    task4 = export_vehicle_detect_model(base_model=base_model)
    task4.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit("1")
    task4.set_cpu_limit("4")
    task4.set_memory_limit("16Gi")
    kfp.kubernetes.add_pod_annotation(
        task=task4,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    # Upload engine artifacts into MLFlow S3 bucket with Triton Compatible directory
    task5 = upload_engines(
        mlflow_url=mlflow_url,
        mlflow_s3_url=mlflow_s3_url,
        license_engine=task3.outputs['engine'],
        license_config=task3.outputs['license_config'],
        vehicle_engine=task4.outputs['engine'],
        vehicle_config=task4.outputs['vehicle_config']
    )
    kfp.kubernetes.add_pod_annotation(
        task=task5,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    return task5.output

In [13]:
base_model = "yolo11s"
# datasets_pvc_name = 'roboflow-license-plate-datasets-e2e'
datasets_pvc_name = '15d4f841-9c2e-4b1b-894f-2a96660b43acroboflow-lp-datasets'
mlflow_url = "https://mlflow.ingress.pcai0308.sg2.hpecolo.net"
mlflow_s3_url = "http://local-s3-service.ezdata-system.svc.cluster.local:30000"
mlflow_experiment = 'license_plate_yolo11s_finetune'

kfp_client.create_run_from_pipeline_func(
    prepare_model_pipe,
    arguments={
        'base_model': base_model,
        'datasets_pvc_name': datasets_pvc_name,
        'epochs': 1,
        'mlflow_url': mlflow_url,
        'mlflow_s3_url': mlflow_s3_url,
        'mlflow_experiment': mlflow_experiment,
    },
    experiment_name="test-rhgt-exp",
)

RunPipelineResult(run_id=cfb30aad-9c5e-41ca-934e-6606ef5426c9)

In [28]:
from kfp import compiler, dsl

compiler.Compiler().compile(preparing_model_pipeline, package_path='preparing_model_pipeline.yaml')